# Quick experiment: use the last 24 hours to predict the next hour

**Question:** Can an MLP forecast the household's next-hour `Global_active_power` (kW) using the last 24 hourly observations?

This is a separate, short experiment. It uses the same hourly cleaning approach, seven lagged measured features and four calendar features as Stage 1, with a chronological 60/20/20 split. All scaling is fitted on training data only. The MLP is compared with last-hour persistence and the value at the same hour yesterday. Its test metrics are reported after training and validation, but this dataset's test period was already viewed in Stage 1, so it is not an untouched holdout. Stage 1's original 168-hour, 24-output models address a different task and are not directly comparable to these scores.

**Runtime:** Up to 100 epochs with patience 10; early stopping may end training sooner. Runtime depends on the kernel and computer. Put this notebook in the project folder with `household_power_consumption.csv` or `household_power_consumption.txt`, and select the `4010-DeepLearning-gpu` kernel.

In [ ]:
from pathlib import Path
import random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '| GPU:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'not available')
print('PyTorch:', torch.__version__)

## Prepare hourly data and avoid leakage

Use hourly averages where at least 30 target minutes are present. Fill short gaps by forward filling, and longer gaps using the previous week when available, followed by forward fill. Every input to a forecast ends before the forecast hour.

In [ ]:
csv_path = Path('household_power_consumption.csv')
txt_path = Path('household_power_consumption.txt')
if csv_path.exists():
    df = pd.read_csv(csv_path)
elif txt_path.exists():
    df = pd.read_csv(txt_path, sep=';', na_values='?')
else:
    raise FileNotFoundError('Put household_power_consumption.csv or household_power_consumption.txt beside this notebook.')
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df = df.drop(columns=['Date', 'Time']).set_index('Datetime')
TARGET, WEEK = 'Global_active_power', 168
counts = df[TARGET].resample('h').count()
hourly_raw = df.resample('h').mean()
hourly_raw.loc[counts < 30] = np.nan
missing = hourly_raw[TARGET].isna()
run_id = (missing != missing.shift()).cumsum()
short_gap = missing & (missing.groupby(run_id).transform('size') <= 3)
hourly = hourly_raw.copy()
hourly.loc[short_gap] = hourly_raw.ffill().loc[short_gap]
for _ in range(2):
    hourly = hourly.fillna(hourly.shift(WEEK))
hourly = hourly.ffill().dropna()
assert not hourly.isna().any().any()
features = hourly.copy()
features['hour_sin'] = np.sin(2 * np.pi * features.index.hour / 24)
features['hour_cos'] = np.cos(2 * np.pi * features.index.hour / 24)
features['dow_sin'] = np.sin(2 * np.pi * features.index.dayofweek / 7)
features['dow_cos'] = np.cos(2 * np.pi * features.index.dayofweek / 7)
FEATURES = list(features.columns)
TARGET_IDX = FEATURES.index(TARGET)
LOOKBACK = 24
n = len(features)
train_end, val_end = int(n * .6), int(n * .8)
scaler = StandardScaler().fit(features.iloc[:train_end])
scaled = scaler.transform(features).astype(np.float32)

def make_windows(start, end):
    origins = np.arange(max(start, LOOKBACK), end)
    X = np.stack([scaled[t - LOOKBACK:t] for t in origins])
    y = scaled[origins, TARGET_IDX].reshape(-1, 1)
    return X, y, origins

X_train, y_train, t_train = make_windows(0, train_end)
X_val, y_val, t_val = make_windows(train_end, val_end)
X_test, y_test, t_test = make_windows(val_end, n)
assert t_train[-1] < train_end <= t_val[0] and t_val[-1] < val_end <= t_test[0]
for label, X, t in [('train', X_train, t_train), ('validation', X_val, t_val), ('test', X_test, t_test)]:
    print(f'{label:10s} X {X.shape} | target hours {features.index[t[0]]} to {features.index[t[-1]]}')

## Train a deeper MLP

The model flattens the 24 × 11 input window, then uses three fully connected layers with 512, 256 and 128 units and dropout to predict one scaled value. Adam minimizes MSE; early stopping monitors validation MSE and restores the best weights. These fixed settings are a quick experiment, not a hyperparameter search.

In [ ]:
model = nn.Sequential(nn.Flatten(), nn.Linear(LOOKBACK * len(FEATURES), 512), nn.ReLU(), nn.Dropout(0.1),
                      nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.1),
                      nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()
Xtr, ytr, Xva, yva = [torch.from_numpy(a).to(device) for a in (X_train, y_train, X_val, y_val)]
history = {'train': [], 'val': []}
best_loss, best_state, wait = float('inf'), None, 0
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 100, 10, 128
start = time.perf_counter()
for epoch in range(MAX_EPOCHS):
    model.train(); total = 0.0
    for idx in torch.randperm(len(Xtr), device=device).split(BATCH_SIZE):
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(Xtr[idx]), ytr[idx])
        loss.backward(); optimizer.step()
        total += loss.detach().item() * len(idx)
    model.eval()
    with torch.no_grad():
        val_loss = sum(loss_fn(model(x), y).item() * len(x) for x, y in zip(Xva.split(2048), yva.split(2048))) / len(Xva)
    history['train'].append(total / len(Xtr)); history['val'].append(val_loss)
    print(f'Epoch {epoch+1:2d}: train MSE {history["train"][-1]:.4f}, validation MSE {val_loss:.4f}')
    if val_loss < best_loss:
        best_loss, wait = val_loss, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        wait += 1
        if wait >= PATIENCE:
            print('Early stopping: validation loss did not improve for', PATIENCE, 'epochs.')
            break
model.load_state_dict(best_state)
if device.type == 'cuda': torch.cuda.synchronize()
print(f'Training completed in {time.perf_counter()-start:.1f} s; best epoch {np.argmin(history["val"])+1}.')
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(history['train'], label='train'); ax.plot(history['val'], label='validation')
ax.set(xlabel='Epoch (zero-based)', ylabel='MSE (scaled)', title='Learning curves'); ax.legend(); plt.show()

## Compare forecasts in kW

Persistence predicts the next hour from the last observed hour. The daily baseline uses the observation 24 hours earlier. We report MAE and RMSE on the same forecast hours for both baselines and the MLP.

In [ ]:
def to_kw(z):
    return np.asarray(z) * scaler.scale_[TARGET_IDX] + scaler.mean_[TARGET_IDX]

def forecast(X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(x.to(device)).cpu() for x in torch.from_numpy(X).split(2048)]).numpy().ravel()

def metrics(actual, predicted):
    error = predicted - actual
    return {'MAE (kW)': float(np.mean(np.abs(error))), 'RMSE (kW)': float(np.sqrt(np.mean(error ** 2)))}

rows = []
for split, X, y in [('validation', X_val, y_val), ('test', X_test, y_test)]:
    actual = to_kw(y.ravel())
    forecasts = {'Persistence': to_kw(X[:, -1, TARGET_IDX]),
                 'Same hour yesterday': to_kw(X[:, 0, TARGET_IDX]),
                 'MLP': to_kw(forecast(X))}
    for name, pred in forecasts.items():
        assert pred.shape == actual.shape and np.isfinite(pred).all()
        rows.append({'split': split, 'model': name, **metrics(actual, pred)})
results = pd.DataFrame(rows).set_index(['split', 'model'])
display(results.round(4))

# A fixed seven-day slice of one-hour-ahead rolling forecasts.
N_PLOT = min(168, len(X_test)); actual = to_kw(y_test[:N_PLOT].ravel())
pred = to_kw(forecast(X_test[:N_PLOT]))
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(features.index[t_test[:N_PLOT]], actual, label='Actual', color='black', lw=1.5)
ax.plot(features.index[t_test[:N_PLOT]], pred, label='MLP', alpha=.85)
ax.set(title='One-hour-ahead rolling forecasts (first week of test period)', ylabel='Global active power (kW)', xlabel='Time')
ax.legend(); plt.tight_layout(); plt.show()